# 06 — Final Model Evaluation & Comparison

Run this notebook only after the baseline, LSTM, and GRU artifacts are available. This is the final held-out test evaluation. Avoid further model tuning based on these results.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from src.config import PROCESSED_DIR, MODEL_DIR, METRICS_DIR, CLASS_NAMES, MAX_VOCAB_SIZE
from src.sequence_data import load_tokenizer, load_sequence_config, texts_to_padded_sequences
from src.evaluate import classification_metrics

METRICS_DIR.mkdir(parents=True, exist_ok=True)
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
y_test = test_df["label_id"].astype(int).to_numpy()
test_text = test_df["review_clean"].astype(str)
print("Test rows:", len(test_df))
display(test_df["sentiment_norm"].value_counts().to_frame("count"))


## 1. Load the shared tokenizer and prepare test sequences

In [ ]:
tokenizer = load_tokenizer(PROCESSED_DIR / "tokenizer.json")
sequence_config = load_sequence_config(PROCESSED_DIR / "sequence_config.json")
X_test = texts_to_padded_sequences(test_text, tokenizer, int(sequence_config["max_length"]))


## 2. Load available models and calculate test predictions

All models are evaluated against the same `y_test` and test rows.

In [ ]:
predictions = {}
model_details = {}

# TF-IDF baseline
baseline_path = MODEL_DIR / "tfidf_logreg.joblib"
if baseline_path.exists():
    baseline = joblib.load(baseline_path)
    predictions["TF-IDF + Logistic Regression"] = baseline.predict(test_text)
    model_details["TF-IDF + Logistic Regression"] = {"parameters": None, "training_seconds": None}

# Recurrent models
for model_type in ["lstm", "gru"]:
    model_path = MODEL_DIR / f"{model_type}_best.keras"
    if model_path.exists():
        model = tf.keras.models.load_model(model_path)
        probabilities = model.predict(X_test, verbose=0)
        predictions[model_type.upper()] = probabilities.argmax(axis=1)
        config_path = METRICS_DIR / f"{model_type}_config.json"
        config = json.loads(config_path.read_text(encoding="utf-8")) if config_path.exists() else {}
        model_details[model_type.upper()] = {
            "parameters": int(model.count_params()),
            "training_seconds": config.get("training_seconds"),
        }

if not predictions:
    raise FileNotFoundError("No trained model artifacts found. Run notebooks 03, 04, and 05 first.")
print("Models evaluated:", list(predictions))


## 3. Aggregate and per-class metrics

In [ ]:
rows = []
for model_name, y_pred in predictions.items():
    metrics = classification_metrics(y_test, y_pred)
    details = model_details.get(model_name, {})
    rows.append({"model": model_name, **metrics, **details})
    print(f"\n===== {model_name} =====")
    print(classification_report(
        y_test, y_pred, labels=[0, 1, 2], target_names=CLASS_NAMES, zero_division=0
    ))

comparison = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
display(comparison)
comparison.to_csv(METRICS_DIR / "test_model_comparison.csv", index=False)


## 4. Confusion matrices

In [ ]:
for model_name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"Confusion Matrix — {model_name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    safe_name = model_name.lower().replace(" ", "_").replace("+", "and")
    plt.savefig(METRICS_DIR.parent / "figures" / f"confusion_matrix_{safe_name}.png", dpi=160)
    plt.show()


## 5. Interpretation

Discuss the Macro F1 ranking, class-wise precision/recall, especially Neutral, confusion patterns, parameter counts, and training time. Explain limitations. Do not equate the highest accuracy with the best model if Macro F1 and minority-class results disagree.